# LAB | Intro to Machine Learning

**Load the data**

In this challenge, we will be working with Spaceship Titanic data. The data can be found here:

https://raw.githubusercontent.com/data-bootcamp-v4/data/main/spaceship_titanic.csv

Metadata

https://github.com/data-bootcamp-v4/data/blob/main/spaceship_titanic.md

## Resolución

El lab sigue la lección 7.1 paso a paso: mirar los datos, quitar los vacíos, separar variables (`features`) y objetivo (`target`), dividir en entrenamiento y test, entrenar un KNN y medir su exactitud con `score`.

**El problema:** predecir si un pasajero de la nave fue transportado a otra dimensión (`Transported`, verdadero o falso). Es una **clasificación**, porque la respuesta es una categoría y no un número.

> **Resolução (português).** O lab segue a aula 7.1 passo a passo: olhar para os dados, tirar os vazios, separar variáveis (`features`) e objectivo (`target`), dividir em treino e teste, treinar um KNN e medir a exactidão com `score`. O problema é prever se um passageiro foi transportado para outra dimensão (`Transported`, verdadeiro ou falso): é uma **classificação**, porque a resposta é uma categoria e não um número.

In [1]:
#import libraries
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split

In [2]:
spaceship = pd.read_csv("https://raw.githubusercontent.com/data-bootcamp-v4/data/main/spaceship_titanic.csv")
spaceship.head()

,PassengerId,HomePlanet,CryoSleep,Cabin,Destination,Age,VIP,RoomService,FoodCourt,ShoppingMall,Spa,VRDeck,Name,Transported
0,0001_01,Europa,False,B/0/P,TRAPPIST-1e,39.0,False,0.0,0.0,0.0,0.0,0.0,Maham Ofracculy,False
1,0002_01,Earth,False,F/0/S,TRAPPIST-1e,24.0,False,109.0,9.0,25.0,549.0,44.0,Juanna Vines,True
2,0003_01,Europa,False,A/0/S,TRAPPIST-1e,58.0,True,43.0,3576.0,0.0,6715.0,49.0,Altark Susent,False
3,0003_02,Europa,False,A/0/S,TRAPPIST-1e,33.0,False,0.0,1283.0,371.0,3329.0,193.0,Solam Susent,False
4,0004_01,Earth,False,F/1/S,TRAPPIST-1e,16.0,False,303.0,70.0,151.0,565.0,2.0,Willy Santantines,True


**Check the shape of your data**

**Cómo funciona**

1. `.shape` devuelve una tupla con (filas, columnas).

**Por qué así:** antes de nada hay que saber con cuántos datos trabajamos.

**Herramientas:** `shape`.

> **Como funciona (português)**
>
> 1. O `.shape` devolve um tuplo com (linhas, colunas).
>
> **Porquê assim:** antes de mais é preciso saber com quantos dados trabalhamos.

In [3]:
spaceship.shape

(8693, 14)

**➜ Qué nos llevamos:** 8.693 pasajeros y 14 columnas.

> **➜ O que tiramos:** 8.693 passageiros e 14 colunas.

**Check for data types**

**Cómo funciona**

1. `.dtypes` enseña el tipo de cada columna.

**Por qué así:** KNN solo acepta números, así que hay que saber qué columnas ya lo son y cuáles son texto o verdadero/falso.

**Herramientas:** `dtypes`.

> **Como funciona (português)**
>
> 1. O `.dtypes` mostra o tipo de cada coluna.
>
> **Porquê assim:** o KNN só aceita números, por isso é preciso saber que colunas já o são.

In [4]:
spaceship.dtypes

PassengerId         str
HomePlanet          str
CryoSleep        object
Cabin               str
Destination         str
Age             float64
VIP              object
RoomService     float64
FoodCourt       float64
ShoppingMall    float64
Spa             float64
VRDeck          float64
Name                str
Transported        bool
dtype: object

**➜ Qué nos llevamos:** solo 6 columnas son números (la edad y los cinco gastos a bordo). `CryoSleep` y `VIP` son verdadero/falso, pero salen como `object` porque tienen vacíos; el resto es texto. El objetivo, `Transported`, es verdadero/falso.

> **➜ O que tiramos:** só 6 colunas são números (a idade e os cinco gastos a bordo). `CryoSleep` e `VIP` são verdadeiro/falso, mas aparecem como `object` porque têm vazios; o resto é texto. O objectivo, `Transported`, é verdadeiro/falso.

**Check for missing values**

**Cómo funciona**

1. `isna().sum()` cuenta los vacíos de cada columna.
2. `isna().any(axis=1)` marca las filas con algún vacío; su suma dice cuántos pasajeros perderíamos al quitarlas.

**Por qué así:** el enunciado dice que hay pocos vacíos y que se pueden quitar las filas; antes conviene comprobar cuántas filas son en total, porque los vacíos de cada columna caen en filas distintas.

**Herramientas:** `isna`, `sum`, `any`, `len`.

> **Como funciona (português)**
>
> 1. O `isna().sum()` conta os vazios de cada coluna.
> 2. O `isna().any(axis=1)` marca as linhas com algum vazio; a soma diz quantos passageiros perderíamos ao tirá-las.
>
> **Porquê assim:** o enunciado diz que há poucos vazios e que se podem tirar as linhas; antes convém verificar quantas linhas são no total, porque os vazios de cada coluna caem em linhas diferentes.

In [5]:
print(spaceship.isna().sum())
filas_con_vacios = spaceship.isna().any(axis=1).sum()
print(f"\nFilas con algún vacío: {filas_con_vacios} de {len(spaceship)} ({filas_con_vacios / len(spaceship):.0%})")

PassengerId       0
HomePlanet      201
CryoSleep       217
Cabin           199
Destination     182
Age             179
VIP             203
RoomService     181
FoodCourt       183
ShoppingMall    208
Spa             183
VRDeck          188
Name            200
Transported       0
dtype: int64

Filas con algún vacío: 2087 de 8693 (24%)


**➜ Qué nos llevamos:** cada columna tiene unos 200 vacíos (2 %), pero caen en filas distintas: al quitar las filas con algún vacío perdemos 2.087 pasajeros, el 24 %. Es más de lo que parece mirando columna a columna, aunque siguen quedando muchos datos.

> **➜ O que tiramos:** cada coluna tem uns 200 vazios (2 %), mas caem em linhas diferentes: ao tirar as linhas com algum vazio perdemos 2.087 passageiros, 24 %. É mais do que parece olhando coluna a coluna, mas continuam a sobrar muitos dados.

There are multiple strategies to handle missing data

- Removing all rows or all columns containing missing data.
- Filling all missing values with a value (mean in continouos or mode in categorical for example).
- Filling all missing values with an algorithm.

For this exercise, because we have such low amount of null values, we will drop rows containing any missing value. 

**Cómo funciona**

1. `dropna()` quita las filas con algún vacío, como pide el enunciado.
2. Se comprueba el tamaño nuevo y que ya no queda ningún vacío.

**Por qué así:** es la opción que indica el lab. Rellenar sería la alternativa si perdiéramos demasiadas filas; con 6.000 pasajeros largos que quedan, hay datos de sobra para un primer modelo.

**Herramientas:** `dropna`, `shape`, `isna`.

> **Como funciona (português)**
>
> 1. O `dropna()` tira as linhas com algum vazio, como pede o enunciado.
> 2. Verifica-se o tamanho novo e que já não há vazios.
>
> **Porquê assim:** é a opção que o lab indica. Preencher seria a alternativa se perdêssemos demasiadas linhas; com mais de 6.000 passageiros, há dados de sobra para um primeiro modelo.

In [6]:
spaceship = spaceship.dropna()
print("Tamaño sin vacíos:", spaceship.shape, "| vacíos que quedan:", spaceship.isna().sum().sum())

Tamaño sin vacíos: (6606, 14) | vacíos que quedan: 0


**➜ Qué nos llevamos:** quedan 6.606 pasajeros sin ningún vacío.

> **➜ O que tiramos:** ficam 6.606 passageiros sem nenhum vazio.

**KNN**

K Nearest Neighbors is a distance based algorithm, and requeries all **input data to be numerical.**

Let's only select numerical columns as our features.

**Cómo funciona**

1. `select_dtypes(include="number")` se queda solo con las columnas numéricas (lección 2.1).
2. Enseña qué columnas han entrado.

**Por qué así:** KNN calcula distancias entre pasajeros, y solo se puede medir la distancia entre números. Las columnas de texto (planeta, cabina, destino...) las convertiremos en el lab de feature engineering.

**Herramientas:** `select_dtypes`, `columns`.

> **Como funciona (português)**
>
> 1. O `select_dtypes(include="number")` fica só com as colunas numéricas (aula 2.1).
> 2. Mostra que colunas entraram.
>
> **Porquê assim:** o KNN calcula distâncias entre passageiros, e só se pode medir a distância entre números. As colunas de texto convertem-se no lab de feature engineering.

In [7]:
features = spaceship.select_dtypes(include="number")
print(list(features.columns))
features.head()

['Age', 'RoomService', 'FoodCourt', 'ShoppingMall', 'Spa', 'VRDeck']


,Age,RoomService,FoodCourt,ShoppingMall,Spa,VRDeck
0,39.0,0.0,0.0,0.0,0.0,0.0
1,24.0,109.0,9.0,25.0,549.0,44.0
2,58.0,43.0,3576.0,0.0,6715.0,49.0
3,33.0,0.0,1283.0,371.0,3329.0,193.0
4,16.0,303.0,70.0,151.0,565.0,2.0


**➜ Qué nos llevamos:** 6 variables: la edad y lo que cada pasajero gastó en servicio de habitaciones, comida, tienda, spa y realidad virtual.

> **➜ O que tiramos:** 6 variáveis: a idade e o que cada passageiro gastou em serviço de quarto, comida, loja, spa e realidade virtual.

And also lets define our target.

**Cómo funciona**

1. El objetivo es la columna `Transported`.
2. `value_counts(normalize=True)` da el porcentaje de cada clase.

**Por qué así:** si una clase fuera mucho más frecuente, un modelo que dijera siempre esa clase ya acertaría mucho; el porcentaje de la clase mayoritaria es el mínimo que hay que superar.

**Herramientas:** selección de columna, `value_counts`.

> **Como funciona (português)**
>
> 1. O objectivo é a coluna `Transported`.
> 2. O `value_counts(normalize=True)` dá a percentagem de cada classe.
>
> **Porquê assim:** se uma classe fosse muito mais frequente, um modelo que dissesse sempre essa classe já acertaria muito; a percentagem da classe maioritária é o mínimo a superar.

In [8]:
target = spaceship["Transported"]
target.value_counts(normalize=True).round(3)

Transported
True     0.504
False    0.496
Name: proportion, dtype: float64

**➜ Qué nos llevamos:** las dos clases están equilibradas (50,4 % transportados). Un modelo que dijera siempre lo mismo acertaría la mitad: ese es el mínimo que hay que superar.

> **➜ O que tiramos:** as duas classes estão equilibradas (50,4 % transportados). Um modelo que dissesse sempre o mesmo acertaria metade: é o mínimo a superar.

**Train Test Split**

Now that we have split the data into **features** and **target** variables and imported the **train_test_split** function, split X and y into X_train, X_test, y_train, and y_test. 80% of the data should be in the training set and 20% in the test set.

**Cómo funciona**

1. `train_test_split` con `test_size=0.20` (80 % entrenamiento, 20 % test) y `random_state=0`, como en la lección.
2. Enseña el tamaño de cada parte.

**Por qué así:** el modelo aprende con el entrenamiento y se evalúa con pasajeros que no ha visto; si lo evaluáramos con los mismos datos con los que aprende, la nota sería engañosa. `random_state` hace que la división sea siempre la misma.

**Herramientas:** `train_test_split`, `shape`.

> **Como funciona (português)**
>
> 1. O `train_test_split` com `test_size=0.20` (80 % treino, 20 % teste) e `random_state=0`, como na aula.
> 2. Mostra o tamanho de cada parte.
>
> **Porquê assim:** o modelo aprende com o treino e avalia-se com passageiros que não viu; avaliar com os mesmos dados com que aprende daria uma nota enganadora. O `random_state` faz que a divisão seja sempre a mesma.

In [9]:
X_train, X_test, y_train, y_test = train_test_split(features, target, test_size=0.20, random_state=0)
print("Entrenamiento:", X_train.shape, "| Test:", X_test.shape)

Entrenamiento: (5284, 6) | Test: (1322, 6)


**➜ Qué nos llevamos:** 5.284 pasajeros para entrenar y 1.322 para el test.

> **➜ O que tiramos:** 5.284 passageiros para treinar e 1.322 para o teste.

**Model Selection**

In this exercise we will be using **KNN** as our predictive model.

You need to choose between **Classificator** or **Regressor**. Take into consideration target variable to decide.

Initialize a KNN instance without setting any hyperparameter.

**Cómo funciona**

1. Como el objetivo es verdadero o falso, se usa `KNeighborsClassifier` (no el regresor).
2. Se crea sin hiperparámetros, como pide el enunciado; por defecto usa 5 vecinos.

**Por qué así:** `KNeighborsRegressor` sirve para predecir números (precios, alquileres); aquí predecimos una categoría.

**Herramientas:** `from sklearn.neighbors import KNeighborsClassifier`.

> **Como funciona (português)**
>
> 1. Como o objectivo é verdadeiro ou falso, usa-se o `KNeighborsClassifier` (não o regressor).
> 2. Cria-se sem hiperparâmetros, como pede o enunciado; por omissão usa 5 vizinhos.
>
> **Porquê assim:** o `KNeighborsRegressor` serve para prever números; aqui prevemos uma categoria.

In [10]:
from sklearn.neighbors import KNeighborsClassifier

knn = KNeighborsClassifier()
print("Vecinos por defecto:", knn.n_neighbors)

Vecinos por defecto: 5


Fit the model to your data.

**Cómo funciona**

1. `fit` entrena el modelo: KNN guarda los pasajeros del entrenamiento para compararlos después.

**Por qué así:** en KNN «entrenar» es memorizar los ejemplos; el trabajo de verdad llega al predecir, cuando busca los 5 pasajeros más parecidos.

**Herramientas:** `fit`.

> **Como funciona (português)**
>
> 1. O `fit` treina o modelo: o KNN guarda os passageiros do treino para os comparar depois.
>
> **Porquê assim:** no KNN «treinar» é decorar os exemplos; o trabalho a sério chega ao prever, quando procura os 5 passageiros mais parecidos.

In [11]:
knn.fit(X_train, y_train)

,"n_neighbors n_neighbors: int, default=5Number of neighbors to use by default for :meth:`kneighbors` queries.",5
,"weights weights: {'uniform', 'distance'}, callable or None, default='uniform'Weight function used in prediction. Possible values:- 'uniform' : uniform weights. All points in each neighborhood are weighted equally.- 'distance' : weight points by the inverse of their distance. in this case, closer neighbors of a query point will have a greater influence than neighbors which are further away.- [callable] : a user-defined function which accepts an array of distances, and returns an array of the same shape containing the weights.Refer to the example entitled:ref:`sphx_glr_auto_examples_neighbors_plot_classification.py`showing the impact of the `weights` parameter on the decisionboundary.",'uniform'
,"algorithm algorithm: {'auto', 'ball_tree', 'kd_tree', 'brute'}, default='auto'Algorithm used to compute the nearest neighbors:- 'ball_tree' will use :class:`BallTree`- 'kd_tree' will use :class:`KDTree`- 'brute' will use a brute-force search.- 'auto' will attempt to decide the most appropriate algorithm based on the values passed to :meth:`fit` method.Note: fitting on sparse input will override the setting ofthis parameter, using brute force.",'auto'
,"leaf_size leaf_size: int, default=30Leaf size passed to BallTree or KDTree. This can affect thespeed of the construction and query, as well as the memoryrequired to store the tree. The optimal value depends on thenature of the problem.",30
,"p p: float, default=2Power parameter for the Minkowski metric. When p = 1, this is equivalentto using manhattan_distance (l1), and euclidean_distance (l2) for p = 2.For arbitrary p, minkowski_distance (l_p) is used. This parameter is expectedto be positive.",2
,"metric metric: str or callable, default='minkowski'Metric to use for distance computation. Default is ""minkowski"", whichresults in the standard Euclidean distance when p = 2. See thedocumentation of `scipy.spatial.distance<https://docs.scipy.org/doc/scipy/reference/spatial.distance.html>`_ andthe metrics listed in:class:`~sklearn.metrics.pairwise.distance_metrics` for valid metricvalues.If metric is ""precomputed"", X is assumed to be a distance matrix andmust be square during fit. X may be a :term:`sparse graph`, in whichcase only ""nonzero"" elements may be considered neighbors.If metric is a callable function, it takes two arrays representing 1Dvectors as inputs and must return one value indicating the distancebetween those vectors. This works for Scipy's metrics, but is lessefficient than passing the metric name as a string.",'minkowski'
,"metric_params metric_params: dict, default=NoneAdditional keyword arguments for the metric function.",None
,"n_jobs n_jobs: int, default=NoneThe number of parallel jobs to run for neighbors search.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary <n_jobs>`for more details.Doesn't affect :meth:`fit` method.",None
Name,Type,Value
"classes_ classes_: array of shape (n_classes,)Class labels known to the classifier","ndarray[bool](2,)","[False, True]"
"effective_metric_ effective_metric_: str or callbleThe distance metric used. It will be same as the `metric` parameteror a synonym of it, e.g. 'euclidean' if the `metric` parameter set to'minkowski' and `p` parameter set to 2.",str,'eu...an'


Evaluate your model.

**Cómo funciona**

1. `score` da la exactitud (accuracy): el porcentaje de pasajeros del test bien clasificados, como en la lección.
2. Se compara con el modelo más simple posible: decir siempre la clase más frecuente del entrenamiento.
3. `pd.crosstab` de real frente a predicho enseña dónde acierta y dónde falla (es lo que en el día 5 se llamará matriz de confusión).

**Por qué así:** una exactitud solo dice algo si se compara con una referencia. La tabla cruzada enseña si el modelo falla más en un sentido que en otro.

**Herramientas:** `score`, `predict`, `value_counts`, `pd.crosstab`.

> **Como funciona (português)**
>
> 1. O `score` dá a exactidão (accuracy): a percentagem de passageiros do teste bem classificados.
> 2. Compara-se com o modelo mais simples possível: dizer sempre a classe mais frequente do treino.
> 3. O `pd.crosstab` de real contra previsto mostra onde acerta e onde falha (é o que no dia 5 se chamará matriz de confusão).
>
> **Porquê assim:** uma exactidão só diz alguma coisa se for comparada com uma referência. A tabela cruzada mostra se o modelo falha mais num sentido do que no outro.

In [12]:
exactitud = knn.score(X_test, y_test)
clase_mayoritaria = y_train.value_counts().index[0]
referencia = (y_test == clase_mayoritaria).mean()
print(f"Exactitud del KNN: {exactitud:.3f}")
print(f"Diciendo siempre «{clase_mayoritaria}»: {referencia:.3f}")

pred = knn.predict(X_test)
pd.crosstab(y_test, pred, rownames=["real"], colnames=["predicho"])

Exactitud del KNN: 0.766
Diciendo siempre «True»: 0.500


predicho,False,True
real,,
False,483,178
True,132,529


**➜ Qué nos llevamos:** con solo la edad y los gastos, el KNN acierta el **76,6 %** de los pasajeros del test, frente al 50 % de decir siempre lo mismo. Se equivoca un poco más dando por transportados a quien no lo fue (178) que al revés (132). Es un primer modelo sin preparar nada; en el lab siguiente añadimos las columnas de texto y la normalización.

> **➜ O que tiramos:** só com a idade e os gastos, o KNN acerta **76,6 %** dos passageiros do teste, contra 50 % de dizer sempre o mesmo. Engana-se um pouco mais a dar como transportado quem não foi (178) do que ao contrário (132). É um primeiro modelo sem preparar nada; no lab seguinte juntamos as colunas de texto e a normalização.

**Congratulations, you have just developed your first Machine Learning model!**